# Notebook 6 — Train, Tune, Evaluate

**Task 2 — From Tables to Notebooks · Qafza Tech MLOps Training 2026/2027**

**Goal:** beat a simple baseline, tune a real model using the validation split, then touch the test set exactly once for the final, honest number.

**What we'll do:**
1. Load the feature tables from Notebook 5
2. Build a simple baseline — the bar every real model has to clear
3. Pick a metric that actually fits an imbalanced problem
4. Train and tune a model using the validation split
5. Evaluate once, on the test set, at the very end
6. Save the model and a results summary


## 1. Load Train / Validation Features

Test features stay unloaded until the last section — no accidental peeking.


In [ ]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

ARTIFACTS_DIR = Path("artifacts/tables")
MODELS_DIR = Path("artifacts/models")
REPORTS_DIR = Path("artifacts/reports")

train_features = pd.read_csv(ARTIFACTS_DIR / "train_features.csv")
val_features = pd.read_csv(ARTIFACTS_DIR / "val_features.csv")

with open(REPORTS_DIR / "feature_list.json") as f:
    FEATURE_COLUMNS = json.load(f)

TARGET = "is_late"

X_train, y_train = train_features[FEATURE_COLUMNS], train_features[TARGET]
X_val, y_val = val_features[FEATURE_COLUMNS], val_features[TARGET]

print("X_train:", X_train.shape, " X_val:", X_val.shape)
print(f"Late rate - train: {y_train.mean():.2%}  val: {y_val.mean():.2%}")


## 2. Choose a Metric That Fits an Imbalanced Problem

With most orders arriving on time, a model that always predicts "on time" would score high **accuracy** while catching zero late deliveries — useless for the actual business question. We need a metric that specifically rewards catching the minority (late) class:

- **Recall** on the late class: of all orders that were actually late, how many did we catch? (Business-relevant: missing a late order means no chance to warn the customer or expedite.)
- **Precision** on the late class: of all orders we flagged as late, how many actually were? (Too many false alarms erode trust in the warning.)
- **Average Precision (PR-AUC)** and **ROC-AUC**: threshold-independent summaries, useful for comparing models/hyperparameters before picking an operating threshold.

We'll track all of them, but use **Average Precision on the validation set** as the single number to decide between hyperparameter settings, since it's more informative than ROC-AUC when the positive class is a minority.


In [ ]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, classification_report, confusion_matrix,
)

def evaluate(y_true, y_pred_proba, y_pred_label, label=""):
    metrics = {
        "average_precision": average_precision_score(y_true, y_pred_proba),
        "roc_auc": roc_auc_score(y_true, y_pred_proba),
        "precision_late": precision_score(y_true, y_pred_label, zero_division=0),
        "recall_late": recall_score(y_true, y_pred_label, zero_division=0),
        "f1_late": f1_score(y_true, y_pred_label, zero_division=0),
    }
    print(f"-- {label} --")
    for k, v in metrics.items():
        print(f"  {k:18s}: {v:.4f}")
    return metrics


## 3. Baseline — the Bar to Beat

The simplest possible baseline: always predict the majority class. If our real model can't beat this on recall for the late class, it isn't adding anything.


In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)

baseline_proba = baseline.predict_proba(X_val)[:, 1]
baseline_pred = baseline.predict(X_val)

baseline_metrics = evaluate(y_val, baseline_proba, baseline_pred, label="Baseline (always predict on-time)")


As expected: this baseline's recall on the late class is 0 — it never flags a single late order. Any model we ship needs to clearly beat this.

## 4. Train and Tune a Real Model

We use a `RandomForestClassifier` with `class_weight="balanced"` so the imbalance is accounted for during training itself, not just at evaluation time. We tune two hyperparameters by hand, training only on `train` and scoring only on `val` — no cross-validation folding the test or validation data back into training.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid = [
    {"n_estimators": 200, "max_depth": 6},
    {"n_estimators": 200, "max_depth": 10},
    {"n_estimators": 400, "max_depth": 10},
    {"n_estimators": 400, "max_depth": None},
]

results = []
for params in param_grid:
    model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    val_proba = model.predict_proba(X_val)[:, 1]
    val_pred = model.predict(X_val)

    ap = average_precision_score(y_val, val_proba)
    results.append({**params, "val_average_precision": ap, "model": model})
    print(f"n_estimators={params['n_estimators']:>4} max_depth={str(params['max_depth']):>4}  ->  val AP = {ap:.4f}")

best = max(results, key=lambda r: r["val_average_precision"])
best_model = best["model"]
print()
print("Best config:", {k: v for k, v in best.items() if k != "model"})


In [ ]:
best_val_proba = best_model.predict_proba(X_val)[:, 1]
best_val_pred = best_model.predict(X_val)

model_val_metrics = evaluate(y_val, best_val_proba, best_val_pred, label="Best RandomForest (validation)")

print()
print("Improvement over baseline (recall on late class):",
      f"{model_val_metrics['recall_late'] - baseline_metrics['recall_late']:+.4f}")


## 5. Touch the Test Set — Once

This is the only cell in the entire task that opens `test_features.csv`. Whatever comes out of this cell is the number we report — no going back to tune further based on it.


In [ ]:
test_features = pd.read_csv(ARTIFACTS_DIR / "test_features.csv")
X_test, y_test = test_features[FEATURE_COLUMNS], test_features[TARGET]

test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = best_model.predict(X_test)

test_metrics = evaluate(y_test, test_proba, test_pred, label="Best RandomForest (TEST - final number)")

print()
print("Classification report:")
print(classification_report(y_test, test_pred, target_names=["on_time", "late"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, test_pred))


## 6. Save the Model and a Results Summary


In [ ]:
import joblib

joblib.dump(best_model, MODELS_DIR / "trained_model.joblib")

summary_lines = [
    "# Notebook 6 Results Summary",
    "",
    f"Best hyperparameters: n_estimators={best['n_estimators']}, max_depth={best['max_depth']}",
    "",
    "## Baseline (validation)",
    *[f"- {k}: {v:.4f}" for k, v in baseline_metrics.items()],
    "",
    "## Best model (validation)",
    *[f"- {k}: {v:.4f}" for k, v in model_val_metrics.items()],
    "",
    "## Best model (TEST - final, touched once)",
    *[f"- {k}: {v:.4f}" for k, v in test_metrics.items()],
]
summary_text = "\n".join(summary_lines)

with open(REPORTS_DIR / "results_summary.md", "w") as f:
    f.write(summary_text)

print(summary_text)
print()
print("Saved model to:", MODELS_DIR / "trained_model.joblib")
print("Saved summary to:", REPORTS_DIR / "results_summary.md")


## Recap — and the Task 2 "Done When" Checklist

- ✅ Six notebooks, each doing one job, run in order from a clean start.
- ✅ Every notebook reads the artifact from the step before it and writes its own for the step after.
- ✅ Split chronologically rather than randomly, with reasoning written down in Notebook 3.
- ✅ First model result compared against a simple baseline, above.

**Artifacts produced by this whole task, sitting in `artifacts/`:**

| File | From |
|---|---|
| `tables/ml_table.csv` | Notebook 1 |
| `tables/labeled_table.csv` | Notebook 2 |
| `tables/train.csv`, `val.csv`, `test.csv` | Notebook 3 |
| `figures/*.png`, `reports/eda_findings.md` | Notebook 4 |
| `tables/train_features.csv`, `val_features.csv`, `test_features.csv`, `models/preprocessor.joblib`, `models/main_product_category_top_values.json`, `reports/feature_list.json` | Notebook 5 |
| `models/trained_model.joblib`, `reports/results_summary.md` | Notebook 6 |

**What's next (a later task, not this one):** turning these six notebooks into clean, production-ready Python scripts that load the same fitted objects and never refit on new data.
